# 第二板块衍生：creditos（信贷板块）

本 Notebook 针对原始数据 `衍生原始数据_331条.csv` 中 `response_body.creditos`（信贷账户列表）做特征衍生。

## 你要求的核心点（口径说明）

- **平铺明细**：把每条 credit 账户记录平铺为一行（`apply_id` 会重复），并取出你列出的字段。
- **窗口过滤日期（你说的“查询日期”）**：本板块按你的要求使用 **`fechaAperturaCuenta（开户日期）`** 做窗口过滤。
  - 定义：
    - `days_before_request_credit = request_time - fechaAperturaCuenta`（单位：天）
    - 在每个窗口 N 天内筛选：`0 <= days_before_request_credit <= N`
- **机构归类**：对 `nombreOtorgante` 按你给的字典归 17 类（新增：个征信机构=SIC；不在字典/缺失 -> 其他）。
- **每窗口/每类别输出**：
  - 该类别数量、总数量、该类别/总数量
  - 对你列的每个“计算数据”都计算：**平均值、标准差、最大值、最小值、notNull占比**
- **新增两套同样口径的分组衍生**：
  - `tipoCuenta`：F/H/R/Q/A/E/P（固定付款/抵押贷款/循环信用/无担保信用/授信或预支信用/分期付款信用/质押信用）
  - `tipoCredito`：CC/PP/TC/LC/PB（消费信贷/个人贷款/信用卡/信用额度/个人银行贷款）
- **异常兜底（A/C/D -> -999）**：沿用第一板块规则：
  - A：关键字段缺失（apply_id/request_time/response_body）
  - C：response_body 非法 JSON
  - D：response_body 里没有 creditos（或 creditos 不是 list）
  命中时该 apply_id 的**所有衍生特征**统一填 `-999`。

## 输出（最终只保留 2 个，默认不写，等你确认后再写）

- `outputs/creditos_flat.csv`：creditos 平铺明细（apply_id 重复）
- `outputs/features_creditos.csv`：按 apply_id 聚合后的特征表


In [1]:
# 这一格负责把原始CSV读进来，把 request_time 统一解析成时间，并准备 A/C/D 的兜底标记（不做复杂衍生）。

# 代码格 1/4：读取数据（逐行注释版）
# 目标：读入 CSV、解析 request_time、并对 A(关键列缺失) 做 -999 兜底准备
# ========================= 这套衍生“到底衍生了什么”（大白话说明）=========================
# 1) 一共衍生了多少个特征？（你要先知道“规模”和“长相”）
#    这套 creditos（信贷板块）本质是：
#    **按时间窗口筛选明细 → 再按不同分组维度（机构大类 / tipoCuenta / tipoCredito）聚合 → 输出数量/占比 + 指标统计。**
#    - 时间窗口：7 个（30/60/90/120/180/360/720 天）
#    - 分组维度一共 3 套：
#    - 每个窗口里会输出这些“板块特征”：
#      [板块A：窗口内总量] 1 个：creditos_{N}d_total_cnt（近 N 天内一共有多少条 creditos 明细）
#      [板块B：机构大类结构] 17 类 × 2 个 = 34 个：
#      [板块C：机构大类数值画像] 17 类 × 16 指标 × 5 统计 = 1360 个
#      [板块D：tipoCuenta 结构] 7 类 × 2 个 = 14 个
#      [板块E：tipoCuenta 数值画像] 7 类 × 16 指标 × 5 统计 = 560 个
#      [板块F：tipoCredito 结构] 5 类 × 2 个 = 10 个
#      [板块G：tipoCredito 数值画像] 5 类 × 16 指标 × 5 统计 = 400 个
#      * 每窗口特征数 = 1 + (34+1360) + (14+560) + (10+400) = 2379
#      * 总特征数 = 2379 × 7 = 16653
#    另外：输出表里会带上 request_time（原始截止时间），它不是衍生特征。
# 3) 核心逻辑（窗口 / 分组 / 时间对比怎么做）
#      * 窗口日期：fechaAperturaCuenta（开户日期，按你要求）
#    - 时间对比的计算怎么做？
#    - 窗口筛选条件：
#    - 分组维度（做三套同口径的分组）：
# 4) 各种缺失/异常怎么处理？你最终会看到 0 / NaN / -999 三种表现
#      => 命中时：该 apply_id 的**所有衍生特征统一 -999**。
#      * 窗口日期 fechaAperturaCuenta 解析失败 -> days_before_request_credit 为空 -> 这条明细会被过滤掉（不进窗口统计）。
# =============================================================================

import json  # 用于解析 response_body 的 JSON 字符串
from pathlib import Path  # 用于更安全地处理文件路径

import numpy as np  # 用于 NaN、数值计算
import pandas as pd  # 用于 DataFrame 处理与聚合

# 你现在要用筛选后的 pickle 数据做第二板块衍生，所以这里增加“优先读 pickle”的逻辑。
# 默认优先读取 cdc_pickle_pass_fpd7.pkl；如果你想回到 CSV，把 USE_PICKLE 改成 False。

USE_PICKLE = True  # True=优先读pickle；False=读CSV

INPUT_PICKLE = Path("cdc_pickle_pass_fpd7.pkl")  # 你筛选后的数据

INPUT_CSV = Path("衍生原始数据_331条.csv")  # 原始 CSV（兜底/回退用）

OUTPUT_DIR = Path("outputs")  # 输出目录（默认不写文件；仅在你打开 WRITE_OUTPUTS 时写）
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # 确保 outputs 目录存在

# 读入数据（优先 pickle；否则读 CSV）。
if USE_PICKLE and INPUT_PICKLE.exists():
    df_raw = pd.read_pickle(INPUT_PICKLE)
    print("[INFO] loaded pickle:", INPUT_PICKLE)
else:
    df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("[INFO] loaded csv:", INPUT_CSV)

# 为了不改变你之前的口径（仍然用 request_time 做窗口截止），这里做一个兼容：apply_time -> request_time。
if ("request_time" not in df_raw.columns) and ("apply_time" in df_raw.columns):
    df_raw["request_time"] = df_raw["apply_time"]
    print("[INFO] mapped apply_time -> request_time")

required_cols = ["apply_id", "request_time", "response_body"]
missing = [c for c in required_cols if c not in df_raw.columns]

# === 情况 A：关键字段缺失 -> 该 apply_id 的衍生特征统一 -999 ===
FATAL_MISSING_COLUMNS = len(missing) > 0

if FATAL_MISSING_COLUMNS:
    print("[WARN] 命中情况A：关键字段缺失 -> 后续该申请的衍生特征将统一填 -999")
    print("[WARN] 缺失字段:", missing)

    # 为了让后续流程还能跑（并输出 -999 特征），补齐缺失列
    if "apply_id" not in df_raw.columns:
        df_raw["apply_id"] = pd.Series(dtype="int64")
    if "request_time" not in df_raw.columns:
        df_raw["request_time"] = pd.NaT
    if "response_body" not in df_raw.columns:
        df_raw["response_body"] = None

# 解析 request_time（解析失败会变 NaT，后面会被当作异常样本处理并置 -999）
df_raw["request_time"] = pd.to_datetime(df_raw["request_time"], errors="coerce")

print("df_raw shape:", df_raw.shape)
df_raw.head(3)


[INFO] loaded pickle: cdc_pickle_pass_fpd7.pkl
[INFO] mapped apply_time -> request_time
df_raw shape: (12546, 12)


,apply_id,response_body,apply_time,approve_state,credit_limit_amount,use_amount,principal_amount_borrowed,fpd7,spd7,credit_apply_cnt,blind_lend,request_time
0,1065991091661283329,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 01:53:44.943,CYCLE_PASS,800.00000000,700.00000000,700.00000000,0,0,1,NaN,2025-11-25 01:53:44.943
1,1066560157648134145,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 06:29:46.980,CYCLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,1.0,2025-11-25 06:29:46.980
2,1066719243236777985,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 07:46:56.302,SINGLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,NaN,2025-11-25 07:46:56.302


In [2]:
# 这一格负责把 response_body.creditos 解析并平铺成明细表，做机构归类、tipo字段标准化、计算窗口用的天数差，以及把日期/数值字段转成可统计的数值。

# 代码格 2/4：处理数据（逐行注释版）
# 目标：
# - 平铺成 creditos_df（apply_id 重复）
# - 机构归类（17 类，新增：个征信机构=SIC）
# - 计算窗口过滤用的“开户日期差”：request_time - fechaAperturaCuenta

# ========== 机构归类字典（与你提供的一致）==========
OTORGANTE_GROUP_DICT = {
    "商店信息": ["TIENDA COMERCIAL", "TIENDA DE AUTOSERVICIO", "TIENDA DEPARTAMENTAL"],
    "大众金融协会": [
        "SOCIEDADES FINANCIERAS POPULARES",
        "SOCIEDAD FINANCIERA COMUNITARIA",
        "ARRENDADORAS FINANCIERAS",
        "UNION DE CREDITO",
    ],
    "多用途": ["SOCIEDAD FINANCIERA DE OBJETO MULTIPLE"],
    "非银行抵押": ["HIPOTECARIO NO BANCARIO"],
    "服务信息": ["SALUD Y SERVICIOS MEDICOS", "SERVICIO MEDICO", "SERVS. GRALES.", "SERVICIOS"],
    "付费电视": ["SERVICIO DE TELEVISION DE PAGA"],
    "个人贷款": ["COMPANIA DE PRESTAMO PERSONAL", "SOFOL PRESTAMO PERSONAL"],
    "基金和信托": ["FONDOS Y FIDEIC", "FONDOS Y FIDEICOMISOS", "FONDOS Y FIDEICO"],
    "建筑": ["MERCANCIA PARA LA CONSTRUCCION"],
    "金融公司": ["SOFOL EMPRESARIAL", "SOFOL AUTOMOTRIZ", "OTRAS FINANCIERA", "ARRENDADORAS NO FINANCIERAS"],
    "通讯": ["TELEFONIA LOCAL Y DE LARGA DISTANCIA", "TELEFONIA CELULAR", "COMUNICACIONES"],
    "销售": ["VENTA POR CATALOGO"],
    "小额贷款": ["MIC CREDITO PERS", "MICROFINANCIERA"],
    "银行": ["BANCOS", "BANCO"],
    "非银行": ["FINANCIERA"],
    "政府": ["GUBERNAMENTALES", "GOBIERNO", "HIPOTECAGOBIERNO"],
    "个征信机构": ["SIC"],
}

GROUPS_TO_USE = list(OTORGANTE_GROUP_DICT.keys())  # 17 个大类

# 机构大类中文名 -> 英文ID（用于特征列名，避免出现中文）
GROUP_NAME_TO_ID = {
    "商店信息": "shop",
    "大众金融协会": "mass_fin_assn",
    "多用途": "multi_purpose",
    "非银行抵押": "nonbank_mortgage",
    "服务信息": "service",
    "付费电视": "paytv",
    "个人贷款": "personal_loan",
    "基金和信托": "fund_trust",
    "建筑": "construction",
    "金融公司": "finance_company",
    "通讯": "telecom",
    "销售": "catalog_sale",
    "小额贷款": "micro_loan",
    "银行": "bank",
    "非银行": "nonbank",
    "政府": "government",
    "个征信机构": "sic",
}
# 英文ID -> 中文名（用于特征字典描述）
GROUP_ID_TO_NAME = {v: k for k, v in GROUP_NAME_TO_ID.items()}

VALUE_TO_GROUP = {raw: g for g, vals in OTORGANTE_GROUP_DICT.items() for raw in vals}  # 反向映射


def map_otorgante_group(nombre_otorgante: str) -> str:
    """把 nombreOtorgante 映射到 17 类（新增：个征信机构=SIC），不在字典/缺失 -> 其他。"""
    if pd.isna(nombre_otorgante):
        return "其他"
    return VALUE_TO_GROUP.get(str(nombre_otorgante), "其他")


# ========== tipoCuenta / tipoCredito 的统计口径（你指定的类别）==========
TIPO_CUENTA_TO_USE = ["F", "H", "R", "Q", "A", "E", "P"]
TIPO_CUENTA_TO_ID = {c: c.lower() for c in TIPO_CUENTA_TO_USE}  # 用于列名（例如 F->f）

TIPO_CREDITO_TO_USE = ["CC", "PP", "TC", "LC", "PB"]
TIPO_CREDITO_TO_ID = {c: c.lower() for c in TIPO_CREDITO_TO_USE}  # 用于列名（例如 CC->cc）


# ========== 解析并平铺 creditos ==========
rows = []

# apply_status：记录每个 apply_id 是否命中 A/C/D（用于最后把该申请的衍生特征统一置为 -999）
apply_status = {aid: "ok" for aid in df_raw["apply_id"].tolist()}

if "FATAL_MISSING_COLUMNS" in globals() and FATAL_MISSING_COLUMNS:
    for aid in apply_status:
        apply_status[aid] = "A_missing_columns"

for apply_id, request_time, response_body in df_raw[["apply_id", "request_time", "response_body"]].itertuples(
    index=False, name=None
):
    if response_body is None or (isinstance(response_body, float) and pd.isna(response_body)) or str(response_body).strip() == "":
        apply_status[apply_id] = "C_bad_json"
        continue

    try:
        obj = json.loads(response_body) if isinstance(response_body, str) else response_body
    except Exception:
        apply_status[apply_id] = "C_bad_json"
        continue

    if not isinstance(obj, dict) or ("creditos" not in obj):
        apply_status[apply_id] = "D_no_creditos_key"
        continue

    creditos = obj.get("creditos")

    if not isinstance(creditos, list):
        apply_status[apply_id] = "D_no_creditos_key"
        continue

    for it in creditos:
        if not isinstance(it, dict):
            continue

        # 平铺字段：按你要求的字段名取出
        rows.append(
            {
                "apply_id": apply_id,
                "request_time": request_time,
                "nombreOtorgante": it.get("nombreOtorgante"),
                "montoPagar": it.get("montoPagar"),
                "saldoActual": it.get("saldoActual"),
                "limiteCredito": it.get("limiteCredito"),
                "saldoVencido": it.get("saldoVencido"),
                "valorActivoValuacion": it.get("valorActivoValuacion"),
                "creditoMaximo": it.get("creditoMaximo"),
                "numeroPagos": it.get("numeroPagos"),
                "peorAtraso": it.get("peorAtraso"),
                "saldoVencidoPeorAtraso": it.get("saldoVencidoPeorAtraso"),
                "fechaUltimaCompra": it.get("fechaUltimaCompra"),
                "fechaAperturaCuenta": it.get("fechaAperturaCuenta"),
                "fechaCierreCuenta": it.get("fechaCierreCuenta"),
                "fechaUltimoPago": it.get("fechaUltimoPago"),
                "fechaPeorAtraso": it.get("fechaPeorAtraso"),
                "tipoCuenta": it.get("tipoCuenta"),
                "tipoCredito": it.get("tipoCredito"),
            }
        )

creditos_df = pd.DataFrame(rows)

print("creditos_df shape (flat):", creditos_df.shape)

# ========== 类型转换（日期/数值）==========
for c in [
    "fechaUltimaCompra",
    "fechaAperturaCuenta",
    "fechaCierreCuenta",
    "fechaUltimoPago",
    "fechaPeorAtraso",
]:
    if c in creditos_df.columns:
        creditos_df[c] = pd.to_datetime(creditos_df[c], errors="coerce")

for c in [
    "montoPagar",
    "saldoActual",
    "limiteCredito",
    "saldoVencido",
    "valorActivoValuacion",
    "creditoMaximo",
    "numeroPagos",
    "peorAtraso",
    "saldoVencidoPeorAtraso",
]:
    if c in creditos_df.columns:
        creditos_df[c] = pd.to_numeric(creditos_df[c], errors="coerce")

# ========== 机构归类 + tipo字段标准化 + 窗口过滤用天数差 ==========
creditos_df["otorgante_group"] = creditos_df["nombreOtorgante"].apply(map_otorgante_group)

creditos_df["tipo_cuenta"] = (
    creditos_df["tipoCuenta"].fillna("").astype(str).str.strip().str.upper()
)
creditos_df["tipo_credito"] = (
    creditos_df["tipoCredito"].fillna("").astype(str).str.strip().str.upper()
)

# ========== dtype 瘦身（路线A）：让大数据更省内存、更稳定 ==========
creditos_df["otorgante_group"] = creditos_df["otorgante_group"].astype("category")
creditos_df["tipo_cuenta"] = creditos_df["tipo_cuenta"].astype("category")
creditos_df["tipo_credito"] = creditos_df["tipo_credito"].astype("category")

for _c in [
    "montoPagar",
    "saldoActual",
    "limiteCredito",
    "saldoVencido",
    "valorActivoValuacion",
    "creditoMaximo",
    "numeroPagos",
    "peorAtraso",
    "saldoVencidoPeorAtraso",
    "days_since_last_payment",
    "days_since_open",
    "days_since_close",
    "is_closed",
    "days_pay_since_open",
    "days_worst_arrears_since_open",
    "days_since_worst_arrears_date",
]:
    if _c in creditos_df.columns:
        creditos_df[_c] = pd.to_numeric(creditos_df[_c], errors="coerce").astype("float32")

# 开户日期差：request_time - fechaAperturaCuenta（天），用于窗口过滤
creditos_df["days_before_request_credit"] = (
    (creditos_df["request_time"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
)

creditos_df_all = creditos_df.copy()

creditos_df = creditos_df[
    creditos_df["days_before_request_credit"].notna() & (creditos_df["days_before_request_credit"] >= 0)
].copy()

# ========== 防穿越：把“晚于 request_time 的日期”统一视为无效（置为 NaT）==========
# 说明：这里只列“第二板块实际平铺且会参与日期差计算”的日期字段；不会混入其它 Notebook（如 BOSS）的字段。
for _c in [
    "fechaCierreCuenta",
    "fechaUltimoPago",
    "fechaPeorAtraso",
    "fechaUltimaCompra",
    "fechaActualizacion",
    "ultimaFechaSaldoCero",
]:
    # 这个 notebook 只会平铺你指定的字段；如果某个日期列根本不存在，就跳过（避免 KeyError）。
    if _c not in creditos_df.columns:
        continue
    creditos_df.loc[
        creditos_df[_c].notna()
        & creditos_df["request_time"].notna()
        & (creditos_df[_c] > creditos_df["request_time"]),
        _c,
    ] = pd.NaT

# ========== 你要的“计算数据”转换为可统计的数值（主要是各种日期差）==========
creditos_df["days_since_last_payment"] = (
    (creditos_df["request_time"] - creditos_df["fechaUltimoPago"]).dt.total_seconds() / 86400
)

creditos_df["days_since_open"] = (
    (creditos_df["request_time"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
)

creditos_df["days_since_close"] = (
    (creditos_df["request_time"] - creditos_df["fechaCierreCuenta"]).dt.total_seconds() / 86400
)

creditos_df["is_closed"] = creditos_df["fechaCierreCuenta"].notna().astype(int)

creditos_df["days_pay_since_open"] = (
    (creditos_df["fechaUltimoPago"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
)

# 数据一致性兜底（非常关键）：如果出现“最后还款日期早于开户日期”，则 days_pay_since_open 会是负数。
# 这种情况在业务含义上不合理（同一账户不应在开户前还款），我们将其视为脏数据并置为 NaN，避免把负值带进统计导致特征异常。
creditos_df.loc[creditos_df["days_pay_since_open"] < 0, "days_pay_since_open"] = np.nan

creditos_df["days_worst_arrears_since_open"] = (
    (creditos_df["fechaPeorAtraso"] - creditos_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
)

# 数据一致性兜底：如果“最严重逾期日期早于开户日期”，则该差值为负。
# 同理视为脏数据并置 NaN，避免统计口径被异常负值污染。
creditos_df.loc[creditos_df["days_worst_arrears_since_open"] < 0, "days_worst_arrears_since_open"] = np.nan

creditos_df["days_since_worst_arrears_date"] = (
    (creditos_df["request_time"] - creditos_df["fechaPeorAtraso"]).dt.total_seconds() / 86400
)

print("creditos_df shape (after cleaning):", creditos_df.shape)
creditos_df.head(5)


creditos_df shape (flat): (697815, 19)
creditos_df shape (after cleaning): (697815, 30)


,apply_id,request_time,nombreOtorgante,montoPagar,saldoActual,limiteCredito,saldoVencido,valorActivoValuacion,creditoMaximo,numeroPagos,...,tipo_cuenta,tipo_credito,days_before_request_credit,days_since_last_payment,days_since_open,days_since_close,is_closed,days_pay_since_open,days_worst_arrears_since_open,days_since_worst_arrears_date
0,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,NaN,7774.0,26.0,...,F,PP,1034.078992,803.078992,1034.078992,803.078992,1,231.0,70.0,964.078992
1,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,8500.0,8500.0,100.0,...,F,PP,701.078992,56.078992,701.078992,56.078992,1,645.0,437.0,264.078992
2,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,84.0,7560.0,0.0,0.0,8400.0,8400.0,100.0,...,F,PP,77.078992,12.078992,77.078992,NaN,0,65.0,44.0,33.078992
3,1065991091661283329,2025-11-25 01:53:44.943,MERCANCIA PARA HOGAR Y OFICINA,0.0,0.0,0.0,0.0,2000.0,4004.0,52.0,...,F,PP,1017.078992,734.078992,1017.078992,734.078992,1,283.0,56.0,961.078992
4,1065991091661283329,2025-11-25 01:53:44.943,SOCIEDAD FINANCIERA DE OBJETO MULTIPLE,0.0,0.0,20000.0,0.0,NaN,600.0,1.0,...,F,PP,294.078992,278.078992,294.078992,278.078992,1,16.0,NaN,NaN


In [3]:
# 这一格把‘明细表’按窗口聚合成‘特征表’：对机构大类 / tipoCuenta / tipoCredito 分组输出 cnt/ratio，并对各指标算 mean/std/min/max/notNull占比，同时对 A/C/D 的申请统一返回 -999。

# 代码格 3/4：衍生函数逻辑（逐行注释版）
# 目标：按窗口（30/60/90/120/180/360/720天）+ 按机构 17 类 聚合统计你要求的所有指标（含新增：个征信机构=SIC）


def _agg_numeric_stats(df: pd.DataFrame, group_cols: list[str], value_col: str) -> pd.DataFrame:
    """对某个数值列做 5 个统计：mean/std/min/max/notnull_ratio。

    - mean/std/min/max：常规聚合
    - notnull_ratio：非空占比 = 非空数量 / 总数量
    """

    total_cnt = df.groupby(group_cols, observed=False).size().rename("__cnt").to_frame()

    notnull_cnt = df.groupby(group_cols, observed=False)[value_col].count().rename("__notnull").to_frame()

    stats = df.groupby(group_cols, observed=False)[value_col].agg(["mean", "std", "min", "max"])  # std 默认 ddof=1

    out = stats.join(total_cnt).join(notnull_cnt)
    out["notnull_ratio"] = (out["__notnull"] / out["__cnt"]).replace([np.inf, -np.inf], np.nan)

    out = out.drop(columns=["__cnt", "__notnull"])

    return out


def derive_creditos_features(
    df_raw: pd.DataFrame,
    creditos_df: pd.DataFrame,
    window_days_list: list[int],
    groups_to_use: list[str],
    tipo_cuenta_to_use: list[str],
    tipo_cuenta_to_id: dict[str, str],
    tipo_credito_to_use: list[str],
    tipo_credito_to_id: dict[str, str],
    apply_status: dict | None = None,
    sentinel_value: float = -999,
) -> pd.DataFrame:
    """对 creditos 板块做特征衍生。

    每个窗口 N 天内：
    - 总数量 total_cnt
    - 机构大类：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）
    - tipoCuenta：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）
    - tipoCredito：每类别数量 cnt / ratio + 指标统计（mean/std/min/max/notnull_ratio）

    注：窗口过滤使用 days_before_request_credit（request_time - fechaAperturaCuenta）。
    """

    df_base = (
        df_raw[["apply_id", "request_time"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")
        .sort_index()
    )

    features = df_base.copy()

    METRIC_COLS = {
        "montoPagar": "monto_pagar",  # 当期应还金额
        "saldoActual": "saldo_actual",  # 当前总待还金额
        "limiteCredito": "limite_credito",  # 信用额度
        "saldoVencido": "saldo_vencido",  # 逾期余额（逾期金额）
        "valorActivoValuacion": "valor_activo_valuacion",  # 资产估值金额
        "creditoMaximo": "credito_maximo",  # 历史最高授信额
        "numeroPagos": "numero_pagos",  # 还款次数
        "peorAtraso": "peor_atraso",  # 最严重逾期期数
        "saldoVencidoPeorAtraso": "saldo_vencido_peor_atraso",  # 最严重逾期时的逾期余额
        "days_since_last_payment": "days_since_last_payment",  # 距付款日期天数（距最后还款日期天数）
        "days_since_open": "days_since_open",  # 距开户日期天数
        "days_since_close": "days_since_close",  # 距账户关闭天数
        "is_closed": "is_closed",  # 账户关闭或者未关闭（0/1）
        "days_pay_since_open": "days_pay_since_open",  # 支付距开户天数（最后还款日-开户日）
        "days_worst_arrears_since_open": "days_worst_arrears_since_open",  # 最严重逾期距开户日期
        "days_since_worst_arrears_date": "days_since_worst_arrears_date",  # 当前日期距离最严重逾期日期
    }

    # 每个窗口生成特征并拼到总 features（路线A：用 dict 收集 + concat，避免 DataFrame fragmentation）
    # 可选：把每个窗口的结果落盘（parquet）以便你调试/复用；默认 False（不额外生成文件）。
    from pathlib import Path
    WRITE_INTERMEDIATE_PARQUET = False
    INTERMEDIATE_DIR = Path("outputs/_tmp_block2_windows")
    if WRITE_INTERMEDIATE_PARQUET:
        INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

    for w in window_days_list:
        sub = creditos_df[
            (creditos_df["days_before_request_credit"] >= 0)
            & (creditos_df["days_before_request_credit"] <= w)
        ].copy()

        # col_dict 用来收集该窗口的所有特征列（避免对 DataFrame 逐列 insert）。
        col_dict: dict[str, pd.Series] = {}

        # 总数量：窗口内所有 creditos 记录数量
        total_cnt = sub.groupby("apply_id", observed=False).size().reindex(df_base.index).fillna(0).astype("int32")
        col_dict[f"creditos_{w}d_total_cnt"] = total_cnt

        denom = total_cnt.replace(0, np.nan)

        # ==================== 机构大类（17类）====================
        sub_cat = sub[sub["otorgante_group"].isin(groups_to_use)].copy()
        cnt = (
            sub_cat.groupby(["apply_id", "otorgante_group"], observed=False).size().unstack().reindex(index=df_base.index, columns=groups_to_use)
        )
        cnt = cnt.fillna(0).astype("int32")
        ratio = cnt.div(denom, axis=0).fillna(0.0).astype("float32")

        for g in groups_to_use:
            gid = GROUP_NAME_TO_ID.get(g, str(g))
            col_dict[f"creditos_{w}d_{gid}_cnt"] = cnt[g]
            col_dict[f"creditos_{w}d_{gid}_ratio"] = ratio[g]

        for raw_col, col_id in METRIC_COLS.items():
            if raw_col not in sub_cat.columns:
                continue
            stats = _agg_numeric_stats(sub_cat, group_cols=["apply_id", "otorgante_group"], value_col=raw_col)
            stats.columns = [f"{col_id}__{c}" for c in stats.columns]

            wide = stats.unstack("otorgante_group").reindex(index=df_base.index)
            wide = wide.reindex(columns=pd.MultiIndex.from_product([stats.columns, groups_to_use]))

            for group_name in groups_to_use:
                gid = GROUP_NAME_TO_ID.get(group_name, str(group_name))
                for stat_name in ["mean", "std", "min", "max", "notnull_ratio"]:
                    key = f"{col_id}__{stat_name}"
                    if (key, group_name) in wide.columns:
                        col_dict[f"creditos_{w}d_{gid}_{col_id}_{stat_name}"] = wide[(key, group_name)].astype("float32")
                    else:
                        col_dict[f"creditos_{w}d_{gid}_{col_id}_{stat_name}"] = pd.Series(np.nan, index=df_base.index, dtype="float32")

        # ==================== 通用：按某个类别字段生成 cnt/ratio + 16指标统计 ====================
        def _add_category_to_dict(*, sub_df: pd.DataFrame, cat_col: str, cat_values: list[str], cat_to_id: dict[str, str], prefix: str) -> None:
            sub_cat_local = sub_df[sub_df[cat_col].isin(cat_values)].copy()

            cnt_local = (
                sub_cat_local.groupby(["apply_id", cat_col], observed=False).size().unstack().reindex(index=df_base.index, columns=cat_values)
            )
            cnt_local = cnt_local.fillna(0).astype("int32")
            ratio_local = cnt_local.div(denom, axis=0).fillna(0.0).astype("float32")

            for v in cat_values:
                vid = cat_to_id.get(v, str(v).lower())
                col_dict[f"creditos_{w}d_{prefix}_{vid}_cnt"] = cnt_local[v]
                col_dict[f"creditos_{w}d_{prefix}_{vid}_ratio"] = ratio_local[v]

            for raw_col, col_id in METRIC_COLS.items():
                if raw_col not in sub_cat_local.columns:
                    continue
                stats_local = _agg_numeric_stats(sub_cat_local, group_cols=["apply_id", cat_col], value_col=raw_col)
                stats_local.columns = [f"{col_id}__{c}" for c in stats_local.columns]

                wide_local = stats_local.unstack(cat_col).reindex(index=df_base.index)
                wide_local = wide_local.reindex(columns=pd.MultiIndex.from_product([stats_local.columns, cat_values]))

                for v in cat_values:
                    vid = cat_to_id.get(v, str(v).lower())
                    for stat_name in ["mean", "std", "min", "max", "notnull_ratio"]:
                        key = f"{col_id}__{stat_name}"
                        if (key, v) in wide_local.columns:
                            col_dict[f"creditos_{w}d_{prefix}_{vid}_{col_id}_{stat_name}"] = wide_local[(key, v)].astype("float32")
                        else:
                            col_dict[f"creditos_{w}d_{prefix}_{vid}_{col_id}_{stat_name}"] = pd.Series(np.nan, index=df_base.index, dtype="float32")

        # === tipoCuenta（7类）===
        _add_category_to_dict(sub_df=sub, cat_col="tipo_cuenta", cat_values=tipo_cuenta_to_use, cat_to_id=tipo_cuenta_to_id, prefix="tipo_cuenta")

        # === tipoCredito（5类）===
        _add_category_to_dict(sub_df=sub, cat_col="tipo_credito", cat_values=tipo_credito_to_use, cat_to_id=tipo_credito_to_id, prefix="tipo_credito")

        window_df = pd.DataFrame(col_dict, index=df_base.index)
        notnull_cols = [c for c in window_df.columns if c.endswith("_notnull_ratio")]
        if len(notnull_cols) > 0:
            window_df[notnull_cols] = window_df[notnull_cols].fillna(0.0).astype("float32")

        # 可选落盘：每个窗口单独保存一个 parquet（默认关闭）。
        if WRITE_INTERMEDIATE_PARQUET:
            window_df.reset_index().to_parquet(INTERMEDIATE_DIR / f"block2_window_{w}d.parquet", index=False)

        # 用 concat 把该窗口结果拼到 features（只拼 7 次，不会碎片化）。
        features = pd.concat([features, window_df], axis=1)

        del window_df, col_dict

    # === A/C/D -> -999（只覆盖衍生特征列，不覆盖 request_time）===
    if apply_status is not None:
        bad_ids = [
            aid
            for aid, st in apply_status.items()
            if st in {"A_missing_columns", "C_bad_json", "D_no_creditos_key"}
        ]
        if len(bad_ids) > 0:
            feature_cols = [c for c in features.columns if c != "request_time"]
            features.loc[bad_ids, feature_cols] = sentinel_value

    return features


In [4]:
# 这一格调用上一格的衍生函数真正生成 features，并用 WRITE_OUTPUTS 开关决定是否把两份CSV写到 outputs（默认不写，等你确认）。

# 代码格 4/4：生成结果（逐行注释版）
# 目标：计算 features 与 creditos_df；默认不落盘 outputs（你确认后再输出两份 CSV）

WINDOW_DAYS = [30, 60, 90, 120, 180, 360, 720]  # 你的时间窗口

features = derive_creditos_features(
    df_raw=df_raw,
    creditos_df=creditos_df,
    window_days_list=WINDOW_DAYS,
    groups_to_use=GROUPS_TO_USE,
    tipo_cuenta_to_use=TIPO_CUENTA_TO_USE,
    tipo_cuenta_to_id=TIPO_CUENTA_TO_ID,
    tipo_credito_to_use=TIPO_CREDITO_TO_USE,
    tipo_credito_to_id=TIPO_CREDITO_TO_ID,
    apply_status=apply_status,
    sentinel_value=-999,
)

# === 删除与第三板块重复的“窗口内总记录数”特征列（只删第二板块）===
# 第二板块与第三板块都在 creditos 上做了 "total_cnt"，为了避免你后面横向合并时列名冲突，这里把第二板块的这 7 列删掉。
_dup_total_cnt_cols = [f"creditos_{w}d_total_cnt" for w in WINDOW_DAYS]
features = features.drop(columns=[c for c in _dup_total_cnt_cols if c in features.columns])

# === 输出开关（默认不输出，等你确认后再改成 True）===
WRITE_OUTPUTS = False

creditos_out_path = OUTPUT_DIR / "creditos_flat.csv"  # 平铺明细
features_out_path = OUTPUT_DIR / "features_creditos.csv"  # 特征表

if WRITE_OUTPUTS:
    creditos_df.to_csv(creditos_out_path, index=False, encoding="utf-8-sig")

    # 只对“衍生特征列”加前缀/后缀（request_time 不加），并写出到 CSV
    _rename_map_for_write = {c: f"cdc_{c}_v2" for c in features.columns if c != "request_time"}

    _features_to_write = features.rename(columns=_rename_map_for_write).reset_index()  # apply_id 变回普通列，便于写 CSV
    _round_cols = [c for c in _features_to_write.columns if c not in {"apply_id", "request_time"}]  # 只对衍生特征列做保留2位小数
    _features_to_write[_round_cols] = _features_to_write[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)  # 统一保留小数点后2位

    _features_to_write.to_csv(
        features_out_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("written:")
    print("-", creditos_out_path)
    print("-", features_out_path)
else:
    print("[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）")

print("creditos_flat rows:", len(creditos_df))
print("features shape:", features.shape)

print("features.request_time NaT rate:", float(features["request_time"].isna().mean()))
print("features.request_time head:", features["request_time"].head(3).tolist())

# === 特征数据字典（不落盘，仅用于核对/拼接）===
# 这里生成一个“特征列名 -> 含义”的映射表 feature_dict，且列格式与另外两板块一致，方便直接上下拼接。

import re


# 把第二板块的特征列名解析成统一的数据字典格式。
def build_feature_dict_block2(cols: list[str]) -> pd.DataFrame:
    # 用 rows 收集每一行“特征定义”。
    rows = []

    # 逐个遍历特征列名。
    for col in cols:
        # request_time 是原始字段，不算衍生特征，所以跳过。
        if col == "request_time":
            continue

        # 窗口总量特征：creditos_{w}d_total_cnt。
        m = re.match(r"^creditos_(\d+)d_total_cnt$", col)
        if m:
            w = int(m.group(1))
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "total",
                    "group_code": "ALL",
                    "metric": "count",
                    "stat": "cnt",
                    "description": f"第二板块：近{w}天内 creditos 总记录数",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_tipo_cuenta_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_cuenta",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第二板块：近{w}天内 tipoCuenta={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_tipo_cuenta_([a-z]+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            metric = m.group(3)
            stat = m.group(4)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_cuenta",
                    "group_code": code,
                    "metric": metric,
                    "stat": stat,
                    "description": f"第二板块：近{w}天内 tipoCuenta={code} 的 {metric} 的 {stat}",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_tipo_credito_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_credito",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第二板块：近{w}天内 tipoCredito={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_tipo_credito_([a-z]+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            metric = m.group(3)
            stat = m.group(4)
            rows.append(
                {
                    "block": "block2_creditos",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_credito",
                    "group_code": code,
                    "metric": metric,
                    "stat": stat,
                    "description": f"第二板块：近{w}天内 tipoCredito={code} 的 {metric} 的 {stat}",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_(.+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            g = m.group(2)
            k = m.group(3)
            if g.startswith("tipo_"):
                pass
            else:
                g_cn = GROUP_ID_TO_NAME.get(g, g)  # 英文ID -> 中文名（用于描述）
                rows.append(
                    {
                        "block": "block2_creditos",
                        "feature_name": col,
                        "window_days": w,
                        "group_type": "otorgante_group",
                        "group_code": g,
                        "metric": "count",
                        "stat": k,
                        "description": f"第二板块：近{w}天内 机构大类={g_cn} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                    }
                )
                continue

        m = re.match(r"^creditos_(\d+)d_(.+)_([a-z0-9_]+)_(mean|std|min|max|notnull_ratio)$", col)
        if m:
            w = int(m.group(1))
            g = m.group(2)
            metric = m.group(3)
            stat = m.group(4)
            if g.startswith("tipo_"):
                pass
            else:
                g_cn = GROUP_ID_TO_NAME.get(g, g)
                rows.append(
                    {
                        "block": "block2_creditos",
                        "feature_name": col,
                        "window_days": w,
                        "group_type": "otorgante_group",
                        "group_code": g,
                        "metric": metric,
                        "stat": stat,
                        "description": f"第二板块：近{w}天内 机构大类={g_cn} 的 {metric} 的 {stat}",
                    }
                )
                continue

        rows.append(
            {
                "block": "block2_creditos",
                "feature_name": col,
                "window_days": None,
                "group_type": "unknown",
                "group_code": None,
                "metric": None,
                "stat": None,
                "description": "第二板块：未匹配到解析规则（请人工确认列名口径）",
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "block",
            "feature_name",
            "window_days",
            "group_type",
            "group_code",
            "metric",
            "stat",
            "description",
        ],
    )


# 生成并展示第二板块的特征字典。
feature_dict = build_feature_dict_block2(list(features.columns))
print("feature_dict shape:", feature_dict.shape)
feature_dict.head(10)

# === 导出：特征字典 CSV（3列：特征名称/中文释义/逻辑解释）===
# 你要的是每个板块各自导出一份“特征字典CSV”，用于你后续手工上下拼接。
# 默认不写文件：你确认后把 WRITE_FEATURE_DICT_OUTPUTS 改成 True 才会写出。
WRITE_FEATURE_DICT_OUTPUTS = True

# 第二板块涉及的“原始字段 -> 中文含义”，用于 logic_desc 解释字段名。
RAW_FIELD_DESC_BLOCK2 = {
    "response_body.creditos": "信贷账户列表",
    "fechaAperturaCuenta": "账户开立日期（开户日期，窗口锚点）",
    "request_time": "截止日期（本次申请时间，来自 apply_time 映射）",
    "nombreOtorgante": "机构类别（原始机构字符串）",
    "tipoCuenta": "账户类型（F/H/R/Q/A/E/P）",
    "tipoCredito": "信用/贷款类型（CC/PP/TC/LC/PB）",
    "montoPagar": "当期应还金额",
    "saldoActual": "当前总待还金额",
    "limiteCredito": "信用额度",
    "saldoVencido": "逾期余额/逾期金额",
    "valorActivoValuacion": "资产估值金额",
    "creditoMaximo": "历史最高授信额",
    "numeroPagos": "还款次数",
    "peorAtraso": "最严重逾期期数",
    "saldoVencidoPeorAtraso": "最严重逾期时的逾期金额",
    "fechaUltimoPago": "最后还款日期",
    "fechaCierreCuenta": "账户关闭日期",
    "fechaPeorAtraso": "最严重逾期日期",
}

METRIC_TO_RAW_FIELD = {
    "monto_pagar": "montoPagar",
    "saldo_actual": "saldoActual",
    "limite_credito": "limiteCredito",
    "saldo_vencido": "saldoVencido",
    "valor_activo_valuacion": "valorActivoValuacion",
    "credito_maximo": "creditoMaximo",
    "numero_pagos": "numeroPagos",
    "peor_atraso": "peorAtraso",
    "saldo_vencido_peor_atraso": "saldoVencidoPeorAtraso",
    "days_since_last_payment": "fechaUltimoPago",
    "days_since_open": "fechaAperturaCuenta",
    "days_since_close": "fechaCierreCuenta",
    "is_closed": "fechaCierreCuenta",
    "days_pay_since_open": "fechaUltimoPago/fechaAperturaCuenta",
    "days_worst_arrears_since_open": "fechaPeorAtraso/fechaAperturaCuenta",
    "days_since_worst_arrears_date": "fechaPeorAtraso",
    "count": "record_count",
}

STAT_DESC_BLOCK2 = {
    "cnt": "数量（该组记录数）",
    "ratio": "占比=该组cnt/窗口内总记录数",
    "mean": "均值=mean(指标值)",
    "std": "标准差=std(指标值)",
    "min": "最小值=min(指标值)",
    "max": "最大值=max(指标值)",
    "notnull_ratio": "非空占比=该指标非空条数/该组总条数",
}

def _logic_desc_block2(r) -> str:
    w = r.get("window_days")
    gtype = r.get("group_type")
    gcode = r.get("group_code")
    metric = r.get("metric")
    stat = r.get("stat")

    # 窗口口径
    window_desc = (
        f"窗口={w}天（锚点fechaAperturaCuenta=账户开立日期；days_before_request_credit=(request_time-fechaAperturaCuenta)/86400；取0<=days<=window）"
    )

    # 分组来源字段说明
    group_desc = (
        f"分组={gtype}:{gcode}（来源字段：nombreOtorgante=机构类别 / tipoCuenta=账户类型 / tipoCredito=信用类型）"
    )

    # 指标来源字段说明
    raw_field = METRIC_TO_RAW_FIELD.get(str(metric), None)
    if raw_field and raw_field in RAW_FIELD_DESC_BLOCK2:
        metric_src = f"指标={metric}（来自{raw_field}={RAW_FIELD_DESC_BLOCK2[raw_field]}）"
    else:
        metric_src = f"指标={metric}"

    stat_explain = STAT_DESC_BLOCK2.get(str(stat), str(stat))

    raw_fields_glossary = (
        "原始字段释义："
        "request_time=截止日期；"
        "fechaAperturaCuenta=开户日期；"
        "montoPagar=当期应还金额；saldoActual=当前总待还金额；limiteCredito=信用额度；saldoVencido=逾期金额；"
        "valorActivoValuacion=资产估值；creditoMaximo=历史最大额度；numeroPagos=还款次数；peorAtraso=最严重逾期期数；saldoVencidoPeorAtraso=最严重逾期金额；"
        "fechaUltimoPago=最后还款日期；fechaCierreCuenta=账户关闭日期；fechaPeorAtraso=最严重逾期日期"
    )

    return f"{window_desc}; {group_desc}; {metric_src}; 统计={stat}（{stat_explain}）; {raw_fields_glossary}"

feature_dict_3col = pd.DataFrame(
    {
        "feature_name": feature_dict["feature_name"],
        "cn_desc": feature_dict["description"],
        "logic_desc": feature_dict.apply(_logic_desc_block2, axis=1),
    }
)

feature_dict_out_path = OUTPUT_DIR / "cdc2_feature.csv"

if WRITE_FEATURE_DICT_OUTPUTS:
    feature_dict_3col.to_csv(feature_dict_out_path, index=False, encoding="utf-8-sig")
    print("written feature_dict:")
    print("-", feature_dict_out_path)
else:
    print("[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）")

# === 特征列名统一加前后缀（只改 features，不改特征字典）===
FEATURE_PREFIX = "cdc_"
FEATURE_SUFFIX = "_v2"

rename_map = {
    c: f"{FEATURE_PREFIX}{c}{FEATURE_SUFFIX}"
    for c in features.columns
    if c != "request_time"
}
features = features.rename(columns=rename_map)

# === 衍生特征统一保留2位小数（只改 features，不改特征字典；request_time 不动）===
_round_cols = [c for c in features.columns if c != "request_time"]
features[_round_cols] = features[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

features

# ===================== 输出 CSV 文件 =====================
# 输出第二板块特征数据到CSV
WRITE_CSV_OUTPUT = True
if WRITE_CSV_OUTPUT:
    # 创建输出DataFrame（features的索引是apply_id，需要reset_index变回列）
    output_df = features.reset_index()
    # 确保列顺序：apply_id, request_time, 其他特征列
    cols = ["apply_id", "request_time"] + [c for c in output_df.columns if c not in ["apply_id", "request_time"]]
    output_df = output_df[cols]
    # 输出到CSV
    csv_path = Path("outputs") / "cdc2_feature_data.csv"
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[WRITE] 第二板块特征数据已输出到: {csv_path.resolve()}")
    print(f"[INFO] 输出数据形状: {output_df.shape}")
    print(f"[INFO] 列顺序: apply_id, request_time, {len(output_df.columns)-2}个特征列")
else:
    print("[INFO] 当前 WRITE_CSV_OUTPUT=False：未输出CSV文件")


[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）
creditos_flat rows: 697815
features shape: (12546, 16647)
features.request_time NaT rate: 0.0
features.request_time head: [Timestamp('2025-11-24 21:37:16.301000'), Timestamp('2025-11-24 21:41:55.655000'), Timestamp('2025-11-24 21:43:14.452000')]
feature_dict shape: (16646, 8)
written feature_dict:
- outputs/cdc2_feature.csv


/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_71736/1158314597.py:390: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output_df = features.reset_index()


[WRITE] 第二板块特征数据已输出到: /Users/zhanglifeng12703/Documents/OverseasPython/CDC/outputs/cdc2_feature_data.csv
[INFO] 输出数据形状: (12546, 16648)
[INFO] 列顺序: apply_id, request_time, 16646个特征列


In [5]:
# - 2）整体 TopIV 特征：等频/等距 分 5 箱（缺失单独一箱），看每箱总量/坏样本/坏率/唯一值数
# - 3）cycle_pass / single_pass 各自 TopIV 特征：同样做等频/等距 5 箱
# - 4）IV>0.1 的特征表：iv、corr_fpd7、na_rate(空值或-999)、psi(后两周vs第一周)、iv_cycle_pass、iv_single_pass
# - 5）“分周 IV”：整体/ cycle_pass / single_pass 维度下，对 Top 特征做每周 IV 计算（用于看稳定性随时间漂移）
# - 默认不写任何文件；如果你确认要落盘，我留了开关 WRITE_REPORTS

# ========== 0. 依赖检查与基础对齐（防止一跑就报错） ==========
import re
import numpy as np
import pandas as pd
from pathlib import Path

# 统一的“空值哨兵”值：A/C/D 兜底用 -999；评估里也把它当空值。
SENTINEL_VALUE = -999

WRITE_REPORTS = True  # 设置为False则不输出cdc2_iv_gt_0.1_report.csv和cdc2_eval_report.xlsx
REPORT_XLSX = Path("outputs") / "cdc2_eval_report.xlsx"

def _safe_div(a: float, b: float) -> float:
    # b=0 时返回 NaN（代表无法计算）。
    return float(a) / float(b) if float(b) != 0 else float("nan")

if "features" not in globals():
    raise RuntimeError("[EVAL] 没找到 features，请先运行上面的衍生结果代码格。")

if "apply_id" in features.columns:
    features = features.set_index("apply_id")

if "request_time" not in features.columns:
    raise RuntimeError("[EVAL] features 缺少 request_time，请先运行上面的衍生结果代码格。")

features["request_time"] = pd.to_datetime(features["request_time"], errors="coerce")

if "df_raw" not in globals():
    raise RuntimeError("[EVAL] 没找到 df_raw，请先运行第1格读取数据代码格。")

_y = (
    df_raw[["apply_id", "fpd7"]]
    .drop_duplicates("apply_id")
    .set_index("apply_id")["fpd7"]
)
_y = pd.to_numeric(_y, errors="coerce")
_y01 = (_y > 0).astype("int8").reindex(features.index)

_state = (
    df_raw[["apply_id", "approve_state"]].drop_duplicates("apply_id").set_index("apply_id")["approve_state"]
    if "approve_state" in df_raw.columns
    else pd.Series("", index=df_raw["apply_id"].drop_duplicates().tolist(), dtype="object")
)
_state = _state.fillna("").astype(str).str.lower().reindex(features.index).fillna("")

mask_cycle = _state.str.contains("cycle_pass", na=False)
mask_single = _state.str.contains("single_pass", na=False)

_t = features["request_time"]
week_start = (_t - pd.to_timedelta(_t.dt.weekday, unit="D")).dt.normalize()

# ========== 1）分周坏率：整体 / cycle_pass / single_pass ==========
def _weekly_bad_rate(mask: pd.Series, name: str) -> pd.DataFrame:
    idx = features.index[mask]
    tmp = pd.DataFrame({"week": week_start.reindex(idx), "y": _y01.reindex(idx)}).dropna(subset=["week", "y"])
    out = tmp.groupby("week", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
    out["sample"] = name
    return out

mask_all = pd.Series(True, index=features.index)

weekly_all = _weekly_bad_rate(mask_all, "overall")
weekly_cycle = _weekly_bad_rate(mask_cycle, "cycle_pass")
weekly_single = _weekly_bad_rate(mask_single, "single_pass")

weekly_bad_rate_df = pd.concat([weekly_all, weekly_cycle, weekly_single], axis=0, ignore_index=True)
weekly_bad_rate_df = weekly_bad_rate_df.sort_values(["week", "sample"], ascending=[True, True])

print("\n[WEEKLY_BAD_RATE] overall/cycle_pass/single_pass:")
print(weekly_bad_rate_df.to_string(index=False))

# ========== 2）基础统计：空值率/同一值率（用于筛选可算 IV 的特征） ==========
# X = 所有衍生特征列（不含 request_time）。
X = features.drop(columns=["request_time"], errors="ignore").copy()

missing_mask = X.isna() | (X == SENTINEL_VALUE)

# 每个特征的空值率。
na_rate = missing_mask.mean(axis=0)

_same = {}
for c in X.columns:
    s = X[c].copy()
    s = s.mask(missing_mask[c], "__MISSING__")
    vc = s.value_counts(dropna=False, normalize=True)
    _same[c] = float(vc.iloc[0]) if len(vc) > 0 else 1.0
same_value_rate = pd.Series(_same).reindex(X.columns)

print("\n[PROFILE] feature_cnt:", int(X.shape[1]))
print("[PROFILE] na_rate>=0.9 cnt:", int((na_rate >= 0.9).sum()))
print("[PROFILE] same_value_rate>=0.99 cnt:", int((same_value_rate >= 0.99).sum()))

# ========== 3）IV 计算（用于 TopIV、IV>0.1 清单、分周IV） ==========
IV_QCUT_BINS = 10

# 单特征 IV。
def _compute_iv_one(x: pd.Series, y01: pd.Series) -> float:
    dfv = pd.DataFrame({"x": x, "y": y01}).dropna(subset=["y"])
    if dfv.shape[0] == 0:
        return float("nan")

    x_raw = dfv["x"]
    y = dfv["y"].astype(int)

    miss = x_raw.isna() | (x_raw == SENTINEL_VALUE)
    x_non = x_raw[~miss]

    if x_non.nunique(dropna=True) <= 2:
        b = x_raw.astype(object).where(~miss, "MISSING")
    else:
        try:
            b_non = pd.qcut(x_non, q=IV_QCUT_BINS, duplicates="drop")
            b = pd.Series("MISSING", index=x_raw.index, dtype="object")
            b.loc[~miss] = b_non.astype(str)
        except Exception:
            b = x_raw.astype(object).where(~miss, "MISSING")

    grp = pd.DataFrame({"b": b, "y": y}).groupby("b", observed=False)["y"].agg(["count", "sum"]).rename(columns={"sum": "bad"})
    grp["good"] = grp["count"] - grp["bad"]

    bad_total = grp["bad"].sum()
    good_total = grp["good"].sum()
    if bad_total == 0 or good_total == 0:
        return 0.0

    k = grp.shape[0]
    grp["bad_dist"] = (grp["bad"] + 0.5) / (bad_total + 0.5 * k)
    grp["good_dist"] = (grp["good"] + 0.5) / (good_total + 0.5 * k)

    woe = np.log(grp["bad_dist"] / grp["good_dist"])
    iv = ((grp["bad_dist"] - grp["good_dist"]) * woe).sum()
    return float(iv)

def _compute_iv_one_subset(x_full: pd.Series, y01_full: pd.Series, mask: pd.Series) -> float:
    return _compute_iv_one(x_full[mask], y01_full[mask])

# 为了可控运行时间，这里只对“非全空/非常量且空值率<0.9”的特征算 IV。
valid_for_iv = (na_rate < 0.9) & (same_value_rate < 0.99)
eligible_cols = X.columns[valid_for_iv].tolist()

TOP_SEARCH_MAX_FEATURES = None
if TOP_SEARCH_MAX_FEATURES is not None:
    eligible_cols = eligible_cols[: int(TOP_SEARCH_MAX_FEATURES)]

print("\n[IV] eligible feature cnt:", int(len(eligible_cols)), "/ total:", int(X.shape[1]))

iv_overall = {}
for c in eligible_cols:
    iv_overall[c] = _compute_iv_one(pd.to_numeric(X[c], errors="coerce"), _y01)
iv_overall_s = pd.Series(iv_overall, name="iv_overall")

feat_top_overall = str(iv_overall_s.sort_values(ascending=False).index[0]) if len(iv_overall_s) > 0 else None
print("[IV] top1 overall:", feat_top_overall, "iv=", float(iv_overall_s.max()) if len(iv_overall_s) > 0 else None)

iv_cycle = {}
if int(mask_cycle.sum()) > 0:
    for c in eligible_cols:
        iv_cycle[c] = _compute_iv_one_subset(pd.to_numeric(X[c], errors="coerce"), _y01, mask_cycle)
    iv_cycle_s = pd.Series(iv_cycle, name="iv_cycle_pass")
    feat_top_cycle = str(iv_cycle_s.sort_values(ascending=False).index[0]) if len(iv_cycle_s) > 0 else None
else:
    iv_cycle_s = pd.Series(dtype="float64", name="iv_cycle_pass")
    feat_top_cycle = None
print("[IV] top1 cycle_pass:", feat_top_cycle, "iv=", float(iv_cycle_s.max()) if len(iv_cycle_s) > 0 else None)

iv_single = {}
if int(mask_single.sum()) > 0:
    for c in eligible_cols:
        iv_single[c] = _compute_iv_one_subset(pd.to_numeric(X[c], errors="coerce"), _y01, mask_single)
    iv_single_s = pd.Series(iv_single, name="iv_single_pass")
    feat_top_single = str(iv_single_s.sort_values(ascending=False).index[0]) if len(iv_single_s) > 0 else None
else:
    iv_single_s = pd.Series(dtype="float64", name="iv_single_pass")
    feat_top_single = None
print("[IV] top1 single_pass:", feat_top_single, "iv=", float(iv_single_s.max()) if len(iv_single_s) > 0 else None)

# ========== 4）TopIV 特征：等频/等距 5 箱（缺失单独一箱） ==========
BIN_N = 5

# 把一个特征做“分箱 + 统计”的通用函数。
def _bin_and_report(feat: str, mask: pd.Series, method: str) -> pd.DataFrame:
    x = pd.to_numeric(X[feat], errors="coerce")
    y = _y01.reindex(x.index)
    x = x[mask]
    y = y[mask]

    miss = x.isna() | (x == SENTINEL_VALUE)
    x_non = x[~miss]

    if x_non.shape[0] == 0:
        bin_label = pd.Series("MISSING", index=x.index, dtype="object")
    else:
        if method == "qcut":
            r = x_non.rank(method="first")
            try:
                b_non = pd.qcut(r, q=BIN_N, duplicates="drop")
                bin_label = pd.Series("MISSING", index=x.index, dtype="object")
                bin_label.loc[~miss] = b_non.astype(str)
            except Exception:
                bin_label = x.astype(object).where(~miss, "MISSING")
        elif method == "cut":
            try:
                b_non = pd.cut(x_non, bins=BIN_N, include_lowest=True)
                bin_label = pd.Series("MISSING", index=x.index, dtype="object")
                bin_label.loc[~miss] = b_non.astype(str)
            except Exception:
                bin_label = x.astype(object).where(~miss, "MISSING")
        else:
            raise ValueError("unknown method: " + str(method))

    box = pd.DataFrame({"bin": bin_label, "y": y}).dropna(subset=["y"])
    out = box.groupby("bin", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)

    x_for_uniq = x.where(~miss, np.nan)
    uniq = (
        pd.DataFrame({"bin": bin_label, "x": x_for_uniq})
        .groupby("bin", observed=False)["x"]
        .nunique(dropna=True)
        .rename("unique_cnt")
        .reset_index()
    )
    out = out.merge(uniq, on="bin", how="left")
    out["unique_cnt"] = out["unique_cnt"].fillna(0).astype(int)

    out["_is_missing"] = (out["bin"] == "MISSING").astype(int)
    out = out.sort_values(["_is_missing", "bin"], ascending=[False, True]).drop(columns=["_is_missing"])

    out.insert(0, "feature", feat)
    out.insert(1, "method", method)
    return out

BIN_TABLES: dict[str, pd.DataFrame] = {}

def _print_top_feat_bins(tag: str, feat: str, mask: pd.Series):
    if not feat:
        print(f"[BIN] {tag}: feat is None, skipped")
        return None, None

    df_q = _bin_and_report(feat, mask, method="qcut")
    df_c = _bin_and_report(feat, mask, method="cut")

    BIN_TABLES[f"bin_{tag}_qcut"] = df_q
    BIN_TABLES[f"bin_{tag}_cut"] = df_c

    print(f"[BIN] {tag} | feature={feat} | equal-frequency(qcut) bins={BIN_N} + missing")
    print(df_q.to_string(index=False))
    print(f"[BIN] {tag} | feature={feat} | equal-width(cut) bins={BIN_N} + missing")
    print(df_c.to_string(index=False))

    return df_q, df_c



_print_top_feat_bins("overall", feat_top_overall, mask_all)
_print_top_feat_bins("cycle_pass", feat_top_cycle, mask_cycle)
_print_top_feat_bins("single_pass", feat_top_single, mask_single)

# ========== 5）IV>0.1 清单：iv/corr/na_rate/psi/iv_cycle/iv_single ==========
# 先取整体 IV（iv_overall_s）里 >0.1 的特征作为候选清单。
IV_THRESHOLD = 0.1
iv_gt = iv_overall_s[iv_overall_s > IV_THRESHOLD].sort_values(ascending=False)

print("\n[IV_GT] IV>0.1 cnt:", int(iv_gt.shape[0]))

# corr_fpd7：用非缺失样本计算 Pearson；缺失口径=NaN 或 -999。
_fpd7_num = (
    df_raw[["apply_id", "fpd7"]].drop_duplicates("apply_id").set_index("apply_id")["fpd7"]
    .pipe(pd.to_numeric, errors="coerce")
    .reindex(features.index)
)

PSI_Q = 10
EPS = 1e-6

_weeks_sorted = sorted([w for w in week_start.dropna().unique()])
_first_w = _weeks_sorted[0] if len(_weeks_sorted) >= 1 else None
_last_two = _weeks_sorted[-2:] if len(_weeks_sorted) >= 2 else _weeks_sorted
_base_mask = (week_start == _first_w) if _first_w is not None else pd.Series(False, index=features.index)
_comp_mask = week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=features.index)

print("\n[PSI] first_week_start:", _first_w)
print("[PSI] last_two_week_start:", _last_two)
print("[PSI] base_cnt:", int(_base_mask.sum()), "comp_cnt:", int(_comp_mask.sum()))

# 单特征 PSI。
def _psi_one(x: pd.Series, base_mask: pd.Series, comp_mask: pd.Series) -> float:
    miss = x.isna() | (x == SENTINEL_VALUE)

    xb = x[base_mask]
    xc = x[comp_mask]
    mb = miss[base_mask]
    mc = miss[comp_mask]

    if xb.shape[0] == 0 or xc.shape[0] == 0:
        return float("nan")

    xb_non = xb[~mb]
    xc_non = xc[~mc]

    if xb_non.nunique(dropna=True) <= 2:
        bb = xb.astype(object).where(~mb, "MISSING")
        bc = xc.astype(object).where(~mc, "MISSING")
    else:
        try:
            _, edges = pd.qcut(xb_non, q=PSI_Q, retbins=True, duplicates="drop")
            edges = sorted(set(edges.tolist()))
            if len(edges) < 3:
                bb = xb.astype(object).where(~mb, "MISSING")
                bc = xc.astype(object).where(~mc, "MISSING")
            else:
                bb_non = pd.cut(xb_non, bins=edges, include_lowest=True)
                bc_non = pd.cut(xc_non, bins=edges, include_lowest=True)
                bb = pd.Series("MISSING", index=xb.index, dtype="object")
                bc = pd.Series("MISSING", index=xc.index, dtype="object")
                bb.loc[~mb] = bb_non.astype(str)
                bc.loc[~mc] = bc_non.astype(str)
        except Exception:
            bb = xb.astype(object).where(~mb, "MISSING")
            bc = xc.astype(object).where(~mc, "MISSING")

    pb = bb.value_counts(normalize=True)
    pc = bc.value_counts(normalize=True)

    cats = list(pb.index.union(pc.index))
    psi = 0.0
    for k in cats:
        p = float(pb.get(k, 0.0)); q = float(pc.get(k, 0.0))
        p = max(p, EPS); q = max(q, EPS)
        psi += (p - q) * float(np.log(p / q))
    return float(psi)

rows = []
for feat in iv_gt.index.tolist():
    x = pd.to_numeric(X[feat], errors="coerce")

    miss = x.isna() | (x == SENTINEL_VALUE)
    na = float(miss.mean())

    valid = (~miss) & _fpd7_num.notna()
    corr = float(pd.Series(x[valid]).corr(_fpd7_num[valid], method="pearson")) if int(valid.sum()) >= 3 else float("nan")

    psi = _psi_one(x, _base_mask, _comp_mask) if (_first_w is not None and len(_last_two) > 0) else float("nan")

    iv_c = _compute_iv_one_subset(x, _y01, mask_cycle) if int(mask_cycle.sum()) > 0 else float("nan")
    iv_sg = _compute_iv_one_subset(x, _y01, mask_single) if int(mask_single.sum()) > 0 else float("nan")

    rows.append(
        {
            "feature_name": feat,
            "iv": float(iv_gt.loc[feat]),
            "corr_fpd7": corr,
            "na_rate_na_or_-999": na,
            "psi_last2w_vs_firstw": psi,
            "iv_cycle_pass": iv_c,
            "iv_single_pass": iv_sg,
        }
    )

iv_report_df = pd.DataFrame(rows)

for c in ["iv", "corr_fpd7", "na_rate_na_or_-999", "psi_last2w_vs_firstw", "iv_cycle_pass", "iv_single_pass"]:
    if c in iv_report_df.columns:
        iv_report_df[c] = pd.to_numeric(iv_report_df[c], errors="coerce").round(4)

iv_report_df = iv_report_df.sort_values("iv", ascending=False).reset_index(drop=True)

print("\n[IV_REPORT] IV>0.1 report (head 50):")
print(iv_report_df.head(50).to_string(index=False))

if WRITE_REPORTS:
    out_csv = Path("outputs") / f"cdc2_iv_gt_{IV_THRESHOLD}_report.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    iv_report_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[WRITE]", str(out_csv.resolve()))

# ========== 6）分周 IV：整体 / cycle_pass / single_pass（默认只算 Top 特征） ==========

def _weekly_iv_for_feature(feat: str, mask: pd.Series) -> pd.DataFrame:
    x = pd.to_numeric(X[feat], errors="coerce")
    y = _y01.reindex(x.index)

    idx = features.index[mask]
    w = week_start.reindex(idx)
    xv = x.reindex(idx)
    yv = y.reindex(idx)

    tmp = pd.DataFrame({"week": w, "x": xv, "y": yv}).dropna(subset=["week", "y"])

    out_rows = []
    for wk, g in tmp.groupby("week", observed=False):
        ivv = _compute_iv_one(g["x"], g["y"])
        out_rows.append({"week": wk, "iv": float(ivv), "total_cnt": int(g.shape[0]), "bad_cnt": int(g["y"].sum())})

    out = pd.DataFrame(out_rows).sort_values("week")
    out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
    out["iv"] = pd.to_numeric(out["iv"], errors="coerce").round(4)
    return out

# 需要做分周 IV 的特征列表（默认 3 个：整体Top、cycleTop、singleTop）。
weekly_iv_targets = []
if feat_top_overall:
    weekly_iv_targets.append(("overall_top", feat_top_overall))
if feat_top_cycle:
    weekly_iv_targets.append(("cycle_top", feat_top_cycle))
if feat_top_single:
    weekly_iv_targets.append(("single_top", feat_top_single))

WEEKLY_IV_TABLES: dict[str, pd.DataFrame] = {}

for tag, feat in weekly_iv_targets:
    df_w_all = _weekly_iv_for_feature(feat, mask_all)
    df_w_cycle = _weekly_iv_for_feature(feat, mask_cycle)
    df_w_single = _weekly_iv_for_feature(feat, mask_single)

    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_overall"] = df_w_all
    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_cycle_pass"] = df_w_cycle
    WEEKLY_IV_TABLES[f"weekly_iv_{tag}_single_pass"] = df_w_single

    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=overall")
    print(df_w_all.to_string(index=False))
    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=cycle_pass")
    print(df_w_cycle.to_string(index=False))
    print(f"[WEEKLY_IV] {tag} | feature={feat} | sample=single_pass")
    print(df_w_single.to_string(index=False))

# 如果你想把“IV>0.1 全部特征”也做分周 IV，会非常慢；这里留一个开关。
WEEKLY_IV_FOR_ALL_IV_GT = False

if WEEKLY_IV_FOR_ALL_IV_GT:
    feats = iv_gt.index.tolist()[: int(WEEKLY_IV_MAX_FEATURES)]
    for feat in feats:
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=overall")
        print(_weekly_iv_for_feature(feat, mask_all).to_string(index=False))
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=cycle_pass")
        print(_weekly_iv_for_feature(feat, mask_cycle).to_string(index=False))
        print(f"\n[WEEKLY_IV_ALL] feature={feat} | sample=single_pass")
        print(_weekly_iv_for_feature(feat, mask_single).to_string(index=False))


# ========== 7）写出一个 Excel：衍生评估报告（多 sheet） ==========

def _safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))[:31] or "sheet"
    s = base
    i = 1
    while s in used:
        suffix = f"_{i}"
        s = (base[: max(0, 31 - len(suffix))] + suffix)
        i += 1
    used.add(s)
    return s

if WRITE_REPORTS:
    REPORT_XLSX.parent.mkdir(parents=True, exist_ok=True)

    tables: dict[str, pd.DataFrame] = {}
    tables["weekly_bad_rate"] = weekly_bad_rate_df

    if "BIN_TABLES" in globals():
        tables.update(BIN_TABLES)

    tables["iv_gt_0p1_report"] = iv_report_df

    # 分周 IV（Top 特征）
    if "WEEKLY_IV_TABLES" in globals():
        tables.update(WEEKLY_IV_TABLES)

    meta = pd.DataFrame(
        {
            "item": [
                "y_definition",
                "missing_definition",
                "week_anchor",
                "bin_n",
                "iv_threshold",
                "top_overall_feature",
                "top_cycle_feature",
                "top_single_feature",
                "cycle_cnt",
                "single_cnt",
                "total_cnt",
            ],
            "value": [
                "y=(fpd7>0)->1 else 0",
                "missing = NaN or -999",
                "week_start = Monday 00:00:00 based on request_time",
                str(BIN_N),
                str(IV_THRESHOLD),
                str(feat_top_overall),
                str(feat_top_cycle),
                str(feat_top_single),
                str(int(mask_cycle.sum())),
                str(int(mask_single.sum())),
                str(int(features.shape[0])),
            ],
        }
    )

    # 写 Excel（优先 openpyxl）。
    used = set()
    try:
        writer = pd.ExcelWriter(REPORT_XLSX, engine="openpyxl")
    except Exception:
        writer = pd.ExcelWriter(REPORT_XLSX)

    with writer:
        meta.to_excel(writer, sheet_name=_safe_sheet_name("meta", used), index=False)
        for k, df in tables.items():
            if df is None:
                df = pd.DataFrame()
            if not isinstance(df, pd.DataFrame):
                df = pd.DataFrame({"value": [str(df)]})
            df.to_excel(writer, sheet_name=_safe_sheet_name(k, used), index=False)

    print("[WRITE_EXCEL] eval report saved:", str(REPORT_XLSX.resolve()))

# =========================
# 额外：生成 quality 检测 Excel（每个特征一行）
# =========================
# 按你的要求，第二板块 notebook 也能独立输出 quality 表。
# 注意：第二板块特征列数非常多（1w+），打开 WRITE_QUALITY_EXCEL 后计算会很慢。

WRITE_QUALITY_EXCEL = False
QUALITY_XLSX = Path("outputs") / "block2_feature_quality.xlsx"

if WRITE_QUALITY_EXCEL:
    import scripts.build_all_blocks_feature_quality_excel as q

    _base = df_raw[["apply_id", "request_time", "fpd7", "approve_state"]].copy()
    _base["apply_id"] = _base["apply_id"].astype(str)
    _base = _base.drop_duplicates("apply_id").set_index("apply_id")

    _y = (pd.to_numeric(_base["fpd7"], errors="coerce") > 0).astype(int)
    _state = _base["approve_state"].fillna("").astype(str).str.lower()
    _is_cycle = _state.str.contains("cycle_pass", na=False)
    _is_single = _state.str.contains("single_pass", na=False)

    _rt = pd.to_datetime(_base["request_time"], errors="coerce")
    _week_start = (_rt - pd.to_timedelta(_rt.dt.weekday, unit="D")).dt.normalize()
    _weeks = sorted([w for w in _week_start.dropna().unique()])
    _first_w = _weeks[0] if len(_weeks) >= 1 else None
    _last_two = _weeks[-2:] if len(_weeks) >= 2 else _weeks
    _base_mask = (_week_start == _first_w) if _first_w is not None else pd.Series(False, index=_week_start.index)
    _comp_mask = _week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=_week_start.index)

    _feat = features.copy()
    if "apply_id" in _feat.columns:
        _feat = _feat.set_index("apply_id")
    _feat.index = _feat.index.astype(str)
    _X = _feat.drop(columns=["request_time"], errors="ignore")

    # 第二板块字典 feature_name 不带前后缀。
    _dict = feature_dict_3col.copy()
    _dict["feature_name"] = _dict["feature_name"].astype(str)
    _dict = _dict.drop_duplicates("feature_name").set_index("feature_name")

    _rows = []
    _total = int(_X.shape[0])

    for _col in _X.columns:
        _col_name = str(_col)
        _base_name = q._strip_feature_name(_col_name)

        _x = pd.to_numeric(_X[_col], errors="coerce")
        _miss = _x.isna() | (_x == q.SENTINEL)
        _na_cnt = int(_miss.sum())
        _na_rate = float(_na_cnt / _total) if _total else float("nan")
        _non = _x[~_miss]
        _nunique = int(_non.nunique(dropna=True))

        _cn = str(_dict["cn_desc"].get(_base_name, "")) if _base_name in _dict.index else ""
        _logic = str(_dict["logic_desc"].get(_base_name, "")) if _base_name in _dict.index else ""

        _corr = float("nan")
        _valid = (~_miss) & _y.notna()
        if int(_valid.sum()) >= 3:
            _corr = q._corr_pearson(_x[_valid].to_numpy(dtype="float64", copy=False), _y[_valid].to_numpy(dtype="float64", copy=False))

        _iv = q._iv_one(_x, _y)
        _psi = q._psi_one(_x, _base_mask, _comp_mask)
        _iv_c = q._iv_one(_x[_is_cycle], _y[_is_cycle])
        _iv_s = q._iv_one(_x[_is_single], _y[_is_single])

        _rows.append(
            {
                "feature_name": _base_name,
                "cn_desc": _cn,
                "logic_desc": _logic,
                "na_cnt": _na_cnt,
                "total_cnt": _total,
                "na_rate": round(_na_rate, 6),
                "nunique": _nunique,
                "corr_y": None if pd.isna(_corr) else round(float(_corr), 6),
                "iv": None if pd.isna(_iv) else round(float(_iv), 6),
                "psi_last2w_vs_firstw": None if pd.isna(_psi) else round(float(_psi), 6),
                "iv_cycle_pass": None if pd.isna(_iv_c) else round(float(_iv_c), 6),
                "iv_single_pass": None if pd.isna(_iv_s) else round(float(_iv_s), 6),
                "notebook": "第二板块衍生.ipynb",
            }
        )

    _quality = pd.DataFrame(_rows)
    QUALITY_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(QUALITY_XLSX) as _w:
        _quality.to_excel(_w, sheet_name="report", index=False)
    print("[WRITE_EXCEL] block2 feature quality saved:", str(QUALITY_XLSX.resolve()))



[WEEKLY_BAD_RATE] overall/cycle_pass/single_pass:
      week  total_cnt  bad_cnt  bad_rate      sample
2025-11-24       2185      420    0.1922  cycle_pass
2025-11-24       4888     2002    0.4096     overall
2025-11-24       2703     1582    0.5853 single_pass
2025-12-01       2023      347    0.1715  cycle_pass
2025-12-01       4500     1738    0.3862     overall
2025-12-01       2477     1391    0.5616 single_pass
2025-12-08       1261      255    0.2022  cycle_pass
2025-12-08       2353      713    0.3030     overall
2025-12-08       1092      458    0.4194 single_pass
2025-12-15        539      129    0.2393  cycle_pass
2025-12-15        805      254    0.3155     overall
2025-12-15        266      125    0.4699 single_pass

[PROFILE] feature_cnt: 16646
[PROFILE] na_rate>=0.9 cnt: 9277
[PROFILE] same_value_rate>=0.99 cnt: 6646

[IV] eligible feature cnt: 7369 / total: 16646
[IV] top1 overall: cdc_creditos_90d_tipo_credito_pp_limite_credito_mean_v2 iv= 0.4107683190457869
[IV] top1

/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divid


[IV_REPORT] IV>0.1 report (head 50):
                                                              feature_name     iv  corr_fpd7  na_rate_na_or_-999  psi_last2w_vs_firstw  iv_cycle_pass  iv_single_pass
                   cdc_creditos_90d_tipo_credito_pp_limite_credito_mean_v2 0.4108    -0.1386              0.2673                0.1501         0.1815          0.1750
                    cdc_creditos_90d_tipo_credito_pp_limite_credito_max_v2 0.4027    -0.1344              0.2673                0.1599         0.1683          0.1836
                   cdc_creditos_180d_tipo_credito_pp_limite_credito_max_v2 0.3920    -0.1287              0.1844                0.1565         0.1573          0.1575
                    cdc_creditos_180d_tipo_credito_pp_saldo_vencido_std_v2 0.3906     0.0589              0.2748                0.1315         0.1960          0.2031
                  cdc_creditos_120d_tipo_credito_pp_limite_credito_mean_v2 0.3890    -0.1393              0.2269                0.14